<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_06_gru_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_06 -  SEQ2ONE - Modelo GRU**

Una **GRU (Gated Recurrent Unit)** es un tipo de red neuronal recurrente (RNN) diseñada para modelar **datos secuenciales**, como series temporales financieras.

En una RNN clásica, cada predicción depende no solo del input actual sino también del **estado oculto previo**, lo que le da “memoria”. Sin embargo, las RNN simples sufren del problema de *vanishing gradients*, lo que dificulta el aprendizaje de dependencias de largo plazo.

Las **GRU** fueron introducidas como una versión simplificada de las LSTM. Según el capítulo sobre RNNs del libro *Machine Learning for Algorithmic Trading* :contentReference[oaicite:0]{index=0}, las unidades con compuertas (*gated units*) permiten regular qué información:

- Se mantiene en memoria  
- Se descarta  
- Se actualiza  

A diferencia de la **LSTM**, la GRU:

- Tiene **menos parámetros**  
- Combina algunas compuertas en una sola estructura  
- Es más eficiente computacionalmente  
- Suele entrenar más rápido  

En el contexto de ventanas intradía en formato 3D `(batch, L, F)`, la GRU es adecuada para:

- Capturar dependencias temporales dentro de la ventana  
- Modelar dinámicas no lineales  
- Mantener una arquitectura más liviana que LSTM  

# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [3]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [4]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [5]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [7]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [8]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de data windows**

In [9]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [10]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [11]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [12]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [13]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

In [14]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths, flatten_X = False)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)
'''
def run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):

    size = window_size

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    for target in targets:
        if verbose:
            print(f"\n[BUILD] L{size} | target = '{target}'")


        # crear SOLO 1 bundle (y aplanar X para Ridge)
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],            # <- SOLO UNO
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,              # <- MLP necesita 2D
        )
'''


'\ndef run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):\n\n    size = window_size\n\n    if verbose:\n        print("\n" + "=" * 80)\n        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")\n        print("=" * 80)\n\n    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]\n    rows = []\n\n    for target in targets:\n        if verbose:\n            print(f"\n[BUILD] L{size} | target = \'{target}\'")\n\n\n        # crear SOLO 1 bundle (y aplanar X para Ridge)\n        (bundle,) = create_bundles(\n            window_size=size,\n            targets=[target],            # <- SOLO UNO\n            windows_paths=windows_paths,\n            scalers_paths=scalers_paths,\n            flatten_X=True,              # <- MLP necesita 2D\n        )\n'

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**


In [19]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [20]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [21]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [24]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Modelo GRU - many to one**

**Idea básica**

La **GRU (Gated Recurrent Unit)** es una red neuronal recurrente diseñada para
modelar **dependencias temporales** en secuencias, manteniendo un estado oculto
que se actualiza dinámicamente mediante compuertas.

Al igual que el LSTM, la GRU **no aplana la ventana**, sino que procesa la
secuencia histórica **paso a paso**, preservando el orden temporal de los datos.
Sin embargo, utiliza una arquitectura más simple, con menos parámetros,
al integrar el mecanismo de memoria directamente en el estado oculto.

En configuración **many-to-one**, el modelo recibe una secuencia histórica
y produce un único valor escalar futuro.

Formalmente, el modelo puede expresarse como:

$$
h_t = \mathrm{GRU}(x_t, h_{t-1})
$$

$$
\hat{y}_t = W_o h_T + b_o
$$

donde:
- $x_t \in \mathbb{R}^{20}$ es el vector de features en el minuto $t$,
- $h_t$ es el estado oculto de la GRU,
- $h_T$ resume toda la ventana histórica (por ejemplo, 60 minutos),
- $W_o, b_o$ son los parámetros de la capa de salida.

A diferencia del LSTM, la GRU no mantiene un estado de celda separado;
la memoria relevante queda integrada directamente en $h_t$.

---

**Regularización (GRU)**

**Riesgo:** Medio, ligeramente menor que LSTM debido a su menor complejidad paramétrica.

La regularización **no es automática** y debe controlarse explícitamente:

- **Early stopping:**
  - Mecanismo principal para evitar sobreajuste.
- **Control del tamaño del estado oculto:**
  - Hidden size moderado.
- **Número de capas limitado:**
  - 1 (máximo 2) capas GRU.
- **Dropout (opcional):**
  - Aplicado entre capas, no dentro de la recurrencia.

La regularización en GRU es principalmente **estructural y temporal**, similar
al LSTM, pero con menor carga paramétrica.

---

**Por qué la GRU es relevante en este proyecto**

- Entrada **secuencial explícita**: 60 × 20 (minutos × features).
- Capacidad para capturar:
  - dependencias temporales,
  - dinámica intradía,
  - interacciones no lineales a lo largo del tiempo.
- Modelo:
  - más expresivo que MLP,
  - más liviano que LSTM,
  - más eficiente computacionalmente.

La GRU permite explotar directamente la estructura temporal del problema,
manteniendo la naturaleza secuencial del pipeline con una arquitectura
más compacta que LSTM.

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Tipo: GRU many-to-one
- Número de capas: 1
- Dimensión del estado oculto: moderada (por ejemplo, 64–128)
- Dropout: desactivado inicialmente
- Optimización: Adam
- Early stopping: activado
- **Sin validación interna automática** (la evaluación se realiza externamente en VALID)

El ajuste fino de la arquitectura y la regularización se aborda en etapas posteriores.


### **10.2. Imports (PyTorch) + semillas**

In [25]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### **10.3. DataLoaders desde bundle (con reshape interno)**

In [26]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_gru_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 16384,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para LSTM many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#loaders_lstm_60 = make_lstm_loaders_from_bundle(bundle_60, seq_len=60, n_features=20, batch_size=16384)
#loaders_lstm_90 = make_lstm_loaders_from_bundle(bundle_90, seq_len=60, n_features=20, batch_size=16384)


### **10.4. Modelo LSTM many-to-one**

In [28]:
import torch
import torch.nn as nn

class GRUSeq2One(nn.Module):
    """
    GRU many-to-one:
      X: (B, L, F)
      y: (B, 1)
    Usa el último hidden state como representación.
    """

    def __init__(
        self,
        *,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 1,
        dropout: float = 0.0,   # solo aplica si num_layers > 1
        head_dropout: float = 0.0,
        bidirectional: bool = False,
    ):
        super().__init__()

        self.bidirectional = bidirectional
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=(dropout if num_layers > 1 else 0.0),
            batch_first=True,
            bidirectional=bidirectional,
        )

        out_dim = hidden_size * (2 if bidirectional else 1)

        self.head = nn.Sequential(
            nn.Dropout(head_dropout),
            nn.Linear(out_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, F)

        _, h_n = self.gru(x)
        # h_n: (num_layers * num_directions, B, hidden_size)

        if self.bidirectional:
            # tomar última capa forward y backward y concatenarlas
            h_forward = h_n[-2]
            h_backward = h_n[-1]
            h_last = torch.cat([h_forward, h_backward], dim=1)
        else:
            h_last = h_n[-1]

        return self.head(h_last)


### **10.5. Train: early stopping + gradient clipping + scheduler**



In [29]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

@torch.no_grad()
def eval_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum, n = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

In [30]:
def train_gru(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    head_dropout: float = 0.0,
    bidirectional: bool = False,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
) -> tuple[nn.Module, dict]:
    model = GRUSeq2One(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        head_dropout=head_dropout,
        bidirectional=bidirectional,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=2, min_lr=1e-5
        )

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    history = {"best_valid_mse": None, "epochs_ran": 0, "final_lr": None}

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad_norm))

            opt.step()

        valid_mse = eval_mse(model, loaders["valid"], device)
        if scheduler is not None:
            scheduler.step(valid_mse)

        current_lr = opt.param_groups[0]["lr"]
        print(f"epoch={epoch:02d} | valid_mse={valid_mse:.6f} | lr={current_lr:.2e}")

        history["epochs_ran"] = epoch
        history["final_lr"] = float(current_lr)

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}). Best valid_mse={best_valid:.6f}")
                break

    history["best_valid_mse"] = float(best_valid)
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


### **10.6. Predicción GRU (VALID/TEST) desde X_flat**


In [31]:
@torch.no_grad()
def predict_gru(
    model: nn.Module,
    X_seq: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 4096,
) -> np.ndarray:
    """
    Predicción para GRU many-to-one.

    Espera:
        X_seq: (n, seq_len, n_features)
    Retorna:
        preds: (n,)
    """
    model.eval()

    if X_seq.ndim != 3:
        raise ValueError(f"Se esperaba X 3D (n, L, F). Recibido shape={X_seq.shape}")

    n = X_seq.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_seq[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).squeeze(-1)
        preds.append(yb.detach().cpu().numpy())

        del xb, yb

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(preds, axis=0)


In [32]:
import numpy as np

def get_metrics_torch(
    bundle: dict,
    model,
    *,
    device,
    predict_fn,
    batch_size_pred: int = 32768,
    compute_r2: bool = True,
) -> tuple[dict, dict]:
    """
    Calcula métricas valid/test para modelos seq2one.
    - predict_fn debe devolver y_pred con shape (n,) (o convertible a 1D).
    - bundle['valid'/'test']['X'] puede ser 2D (MLP) o 3D (LSTM).
    """
    # -------- VALID --------
    X_valid = np.asarray(bundle["valid"]["X"], dtype=np.float32)
    y_valid = np.asarray(bundle["valid"]["y"], dtype=np.float32).reshape(-1)

    y_pred_valid = predict_fn(model, X_valid, device=device, batch_size=batch_size_pred)
    y_pred_valid = np.asarray(y_pred_valid).reshape(-1)

    metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=compute_r2)

    # -------- TEST --------
    X_test = np.asarray(bundle["test"]["X"], dtype=np.float32)
    y_test = np.asarray(bundle["test"]["y"], dtype=np.float32).reshape(-1)

    y_pred_test = predict_fn(model, X_test, device=device, batch_size=batch_size_pred)
    y_pred_test = np.asarray(y_pred_test).reshape(-1)

    metrics_test = compute_seq2one_metrics(y_test, y_pred_test, compute_r2=compute_r2)

    return metrics_valid, metrics_test

## **11. Ejecución completa**

In [37]:
import pandas as pd
import numpy as np
import torch
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")

def run_gru(
    window_size: int,
    *,
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 2048,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    verbose: bool = True,
):
    L = int(window_size)

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] GRU | SEQ2ONE | WINDOW_SIZE=L{L} | (L,F)=({L},{n_features}) "
            f"| hs={hidden_size} | Ls={num_layers} | do={dropout} | wd={weight_decay}"
        )
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []
    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()

        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{L}")

        bundle = None
        loaders = None
        model = None
        hist = None
        metrics_valid = None
        metrics_test = None

        try:
            # -------------------------
            # BUILD BUNDLE (3D)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=False) ...")
            t0 = time.perf_counter()

            (bundle,) = create_bundles(
                window_size=L,
                targets=[target],
                windows_paths=windows_paths,
                scalers_paths=scalers_paths,
                flatten_X=False,  # GRU necesita 3D
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    xshape = bundle["train"]["X"].shape
                    yshape = bundle["train"]["y"].shape
                    print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

            # -------------------------
            # LOADERS
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [LOADERS] Creando DataLoaders (3D) ...")
            t0 = time.perf_counter()

            loaders = make_gru_loaders_from_bundle_3d(
                bundle,
                seq_len=L,
                n_features=n_features,
                batch_size=batch_size_train,
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    ntr = len(loaders["train"].dataset)
                    nva = len(loaders["valid"].dataset)
                    nte = len(loaders["test"].dataset)
                    print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

            # -------------------------
            # TRAIN
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
            t0 = time.perf_counter()

            model, hist = train_gru(
                loaders,
                n_features=n_features,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
                lr=lr,
                weight_decay=weight_decay,
                max_epochs=max_epochs,
                patience=patience,
                clip_grad_norm=clip_grad_norm,
                use_scheduler=use_scheduler,
                device=device,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

            # liberar TRAIN (opcional)
            if verbose:
                print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
            del bundle["train"]
            gc.collect()

            # -------------------------
            # PRED + METRICS
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [PRED] Predicciones + métricas (valid/test) ...")
            t0 = time.perf_counter()

            metrics_valid, metrics_test = get_metrics_torch(
                bundle,
                model,
                device=device,
                predict_fn=predict_gru,
                batch_size_pred=batch_size_pred,
                compute_r2=True,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

            # -------------------------
            # DF APPEND
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [DF] Agregando filas a la tabla ...")
            t0 = time.perf_counter()

            df_v = metrics_to_df(
                metrics_valid,
                model="gru",
                split="valid",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            df_t = metrics_to_df(
                metrics_test,
                model="gru",
                split="test",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            for df_ in (df_v, df_t):
                df_["hidden_size"] = hidden_size
                df_["num_layers"] = num_layers
                df_["dropout"] = dropout
                df_["lr"] = lr
                df_["weight_decay"] = weight_decay
                if isinstance(hist, dict):
                    df_["best_valid_mse"] = hist.get("best_valid_mse")
                    df_["epochs_ran"] = hist.get("epochs_ran")
                    df_["final_lr"] = hist.get("final_lr")

            rows.append(df_v)
            rows.append(df_t)

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

            if verbose:
                dt_target = time.perf_counter() - t_target
                print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={dt_target:.2f}s")

        finally:
            if verbose:
                print(f"[{_ts()}]   [CLEAN] Liberando objetos ...")

            bundle = loaders = model = hist = metrics_valid = metrics_test = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_gru_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_gru_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(df_gru_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    return df_gru_metrics


In [38]:
import pandas as pd

def run_gru_incremental(
    window_sizes: list[int],
    *,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    name: str = "gru",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Corre GRU seq2one incremental por window_size, guardando checkpoints.
    """

    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    base_cols = ["model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"]
    hp_cols = ["hidden_size","num_layers","dropout","lr","w_decay","max_epochs","patience","clip_grad_norm","use_scheduler"]

    if df_all.empty:
        df_all = pd.DataFrame(columns=base_cols + hp_cols)

    for c in base_cols + hp_cols:
        if c not in df_all.columns:
            df_all[c] = pd.NA

    key_cols = [
        "model",
        "hidden_size","num_layers","dropout","lr","w_decay",
        "max_epochs","patience","clip_grad_norm","use_scheduler",
        "window_size","target","split","horizon_min"
    ]

    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    for ws in window_sizes:

        df_ws = df_all[
            (df_all["model"] == "gru") &
            (df_all["hidden_size"] == hidden_size) &
            (df_all["num_layers"] == num_layers) &
            (df_all["dropout"] == dropout) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == weight_decay) &
            (df_all["max_epochs"] == max_epochs) &
            (df_all["patience"] == patience) &
            (df_all["clip_grad_norm"] == clip_grad_norm) &
            (df_all["use_scheduler"] == use_scheduler) &
            (df_all["window_size"] == ws)
        ]

        if len(df_ws) >= 8:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas (gru hs={hidden_size} w_decay={weight_decay}).")
            continue

        if verbose:
            print("\n" + "="*90)
            print(
                f"[RUN] GRU incremental | L{ws} | (L,F)=({ws},36) "
                f"| hs={hidden_size} | Ls={num_layers} | do={dropout} | lr={lr} | w_decay={weight_decay} "
                f"| ep={max_epochs} | pat={patience} | clip={clip_grad_norm} | sched={use_scheduler}"
            )
            print("="*90)

        df_new = run_gru(
            ws,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            clip_grad_norm=clip_grad_norm,
            use_scheduler=use_scheduler,
            verbose=verbose,
        ).copy()

        df_new["hidden_size"] = hidden_size
        df_new["num_layers"] = num_layers
        df_new["dropout"] = dropout
        df_new["lr"] = lr
        df_new["w_decay"] = weight_decay
        df_new["max_epochs"] = max_epochs
        df_new["patience"] = patience
        df_new["clip_grad_norm"] = clip_grad_norm
        df_new["use_scheduler"] = use_scheduler

        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all


In [ ]:
df_gru_all_sizes = run_gru_incremental(
    window_sizes=[30, 60, 90, 120, 180],

    # ---- hiperparámetros estándar GRU ----
    hidden_size=64,
    num_layers=1,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    clip_grad_norm=1.0,
    use_scheduler=True,

    # ---- persistencia ----
    name="gru",   # generará seq2one_gru_metrics.parquet (según tu save)
    verbose=True,
)



[RUN] GRU incremental | L30 | (L,F)=(30,36) | hs=64 | Ls=1 | do=0.0 | lr=0.001 | w_decay=0.0001 | ep=30 | pat=5 | clip=1.0 | sched=True

[01:30:03] GRU | SEQ2ONE | WINDOW_SIZE=L30 | (L,F)=(30,36) | hs=64 | Ls=1 | do=0.0 | wd=0.0001

[01:30:03] [1/4] START target='delta_60' | L30
[01:30:03]   [BUILD] Creando bundle (flatten_X=False) ...
H60 Train: (357504, 30, 36) (357504,)
H60 Valid: (76440, 30, 36) (76440,)
H60 Test : (76832, 30, 36) (76832,)
Scaler H60: StandardScaler
[01:30:14]   [BUILD] OK | train X=(357504, 30, 36) y=(357504,) | dt=11.19s
[01:30:14]   [LOADERS] Creando DataLoaders (3D) ...
[01:30:14]   [LOADERS] OK | n(train/valid/test)=(357504/76440/76832) | dt=0.01s
[01:30:14]   [TRAIN] Iniciando entrenamiento ...
epoch=01 | valid_mse=2876.735139 | lr=1.00e-03
epoch=02 | valid_mse=2785.930462 | lr=1.00e-03
epoch=03 | valid_mse=2700.220480 | lr=1.00e-03
epoch=04 | valid_mse=2641.416529 | lr=1.00e-03
epoch=05 | valid_mse=2603.469630 | lr=1.00e-03
epoch=06 | valid_mse=2578.100909 

In [ ]:
df_gru_all_sizes